In [1]:
# Libraries needed for loading and working with our datasets

import pandas as pd
import numpy as np

In [2]:
# Load the datasets prepared in the previous notebooks

destination_df = pd.read_csv("../data/cleaned/places_features.csv")
weather_df = pd.read_csv("../data/cleaned/weather_complete.csv")
accommodation_df = pd.read_csv("../data/cleaned/accommodation_data.csv")
flights_df = pd.read_csv("../data/cleaned/flights_data.csv")

print("Destination:", destination_df.shape)
print("Weather:", weather_df.shape)
print("Accommodation:", accommodation_df.shape)
print("Flights:", flights_df.shape)

Destination: (50, 12)
Weather: (9, 15)
Accommodation: (4835, 30)
Flights: (90, 20)


In [3]:
# Check the destination-related columns

print("Destination columns:")
print(destination_df.columns.tolist())

print("\nWeather columns:")
print(weather_df.columns.tolist())

print("\nAccommodation columns:")
print(accommodation_df.columns.tolist())

print("\nFlight columns:")
print(flights_df.columns.tolist())

Destination columns:
['destination', 'sight_count', 'park_count', 'restaurant_count', 'water_count', 'forest_count', 'wetland_count', 'river_count', 'mountain_count', 'coastal_count', 'sand_count', 'protected_area_count']

Weather columns:
['destination', 'country', 'latitude', 'longitude', 'temperature', 'feels_like', 'humidity', 'pressure', 'wind_speed', 'cloudiness', 'weather_condition', 'weather_description', 'visibility', 'rain_1h', 'timestamp']

Accommodation columns:
['destination', 'check_in', 'check_out', 'hotel_code', 'hotel_name', 'hotel_category', 'hotel_category_name', 'destination_code', 'destination_name', 'zone_code', 'zone_name', 'hotel_latitude', 'hotel_longitude', 'room_code', 'room_name', 'rate_key', 'rate_class', 'rate_type', 'price', 'allotment', 'payment_type', 'board_code', 'board_name', 'rooms', 'adults', 'children', 'offer_name', 'offer_amount', 'cancellation_amount', 'cancellation_from']

Flight columns:
['origin', 'destination', 'departure_date', 'return_dat

In [4]:
# Check which destinations are available in each dataset

print("Destination:", destination_df["destination"].nunique())
print("Weather:", weather_df["destination"].nunique())
print("Accommodation:", accommodation_df["destination"].nunique())
print("Flights:", flights_df["destination"].nunique())

print("\nAccommodation destinations:")
print(sorted(accommodation_df["destination"].unique()))

print("\nFlight destinations:")
print(sorted(flights_df["destination"].unique()))

Destination: 50
Weather: 9
Accommodation: 39
Flights: 8

Accommodation destinations:
['Agra', 'Ahmedabad', 'Alappuzha', 'Amritsar', 'Bengaluru', 'Bhopal', 'Bhubaneswar', 'Chennai', 'Darjeeling', 'Delhi', 'Dharamshala', 'Gangtok', 'Goa', 'Gokarna', 'Hampi', 'Hyderabad', 'Indore', 'Jaipur', 'Jaisalmer', 'Jodhpur', 'Kochi', 'Kolkata', 'Manali', 'Mumbai', 'Munnar', 'Mussoorie', 'Mysore', 'Nainital', 'Ooty', 'Pondicherry', 'Pune', 'Ranthambore', 'Rishikesh', 'Shimla', 'Srinagar', 'Thiruvananthapuram', 'Udaipur', 'Varanasi', 'Varkala']

Flight destinations:
['BLR', 'BOM', 'COK', 'DEL', 'GOI', 'JAI', 'SXR', 'UDR']


In [12]:
# Map the airport codes used in the flight dataset to destination names

airport_to_destination = {
    "GOI": "Goa",
    "COK": "Munnar",       # COK was used for Munnar in our flight search
    "KUU": "Manali",
    "JAI": "Jaipur",
    "UDR": "Udaipur",
    "JSA": "Jaisalmer",
    "JDH": "Jodhpur",
    "AGR": "Agra",
    "VNS": "Varanasi",
    "DED": "Rishikesh",
    "SLV": "Shimla",
    "IXB": "Darjeeling",
    "CJB": "Ooty",
    "IXE": "Coorg",
    "CCJ": "Wayanad",
    "TRV": "Thiruvananthapuram",
    "MAA": "Chennai",
    "VDY": "Hampi",
    "MYQ": "Mysore",
    "IXZ": "Andaman",
    "BOM": "Mumbai",
    "DEL": "Delhi",
    "ATQ": "Amritsar",
    "IXL": "Ladakh",
    "SXR": "Srinagar",
    "DHM": "Dharamshala",
    "CCU": "Kolkata",
    "BLR": "Bengaluru",
    "HYD": "Hyderabad",
    "PNQ": "Pune",
    "AMD": "Ahmedabad",
    "BHO": "Bhopal",
    "IDR": "Indore",
    "IXR": "Ranchi",
    "BBI": "Bhubaneswar",
    "SHL": "Shillong",
    "JRH": "Kaziranga"
}

# Convert flight airport codes into destination names
flights_df["destination_name"] = flights_df["destination"].map(
    airport_to_destination
)

print("Missing destination mappings:",
      flights_df["destination_name"].isna().sum())

display(
    flights_df[
        ["destination", "destination_name"]
    ].drop_duplicates()
)

Missing destination mappings: 0


,destination,destination_name
0,GOI,Goa
4,COK,Munnar
13,JAI,Jaipur
16,UDR,Udaipur
19,DEL,Delhi
48,BOM,Mumbai
74,SXR,Srinagar
77,BLR,Bengaluru


In [14]:
# Keep the airport code separately and use destination name as the main key

flights_df["airport_code"] = flights_df["destination"]

flights_df["destination"] = flights_df["destination_name"]

flights_df.drop(
    columns=["destination_name"],
    inplace=True
)

print("Flight destinations:")
print(flights_df["destination"].value_counts())

Flight destinations:
destination
Delhi        29
Munnar       18
Mumbai       17
Bengaluru    13
Goa           4
Jaipur        3
Udaipur       3
Srinagar      3
Name: count, dtype: int64


In [15]:
# Create destination-level flight features

flight_features = (
    flights_df
    .groupby("destination")
    .agg(
        flight_count=("destination", "size"),
        min_flight_price=("price", "min"),
        avg_flight_price=("price", "mean"),
        max_flight_price=("price", "max"),
        avg_total_duration=("total_duration", "mean"),
        avg_outbound_stops=("outbound_stops", "mean"),
        avg_return_stops=("return_stops", "mean")
    )
    .reset_index()
)

print("Rows:", len(flight_features))
print("Columns:", flight_features.columns.tolist())

display(flight_features)

Rows: 8
Columns: ['destination', 'flight_count', 'min_flight_price', 'avg_flight_price', 'max_flight_price', 'avg_total_duration', 'avg_outbound_stops', 'avg_return_stops']


,destination,flight_count,min_flight_price,avg_flight_price,max_flight_price,avg_total_duration,avg_outbound_stops,avg_return_stops
0,Bengaluru,13,8776,8776.0,8776,160.384615,0.0,0.000000
1,Delhi,29,14785,14785.0,14785,299.827586,0.0,0.137931
2,Goa,4,9538,9538.0,9538,148.750000,0.0,0.000000
3,Jaipur,3,15767,15767.0,15767,240.000000,0.0,0.000000
4,Mumbai,17,12057,12057.0,12057,190.000000,0.0,0.000000
5,Munnar,18,14132,14132.0,14132,494.444444,1.0,0.555556
6,Srinagar,3,18867,18867.0,18867,705.000000,1.0,1.000000
7,Udaipur,3,20498,20498.0,20498,628.333333,1.0,0.666667


In [ ]:
# Create destination-level accommodation features

accommodation_features = (
    accommodation_df
    .groupby("destination")
    .agg(
        hotel_count=("hotel_code", "nunique"),
        room_count=("room_code", "nunique"),
        min_hotel_price=("price", "min"),
        avg_hotel_price=("price", "mean"),
        max_hotel_price=("price", "max"),
        avg_allotment=("allotment", "mean")
    )
    .reset_index()
)

print("Rows:", len(accommodation_features))
print("Columns:", accommodation_features.columns.tolist())

display(accommodation_features.head(10))


Rows: 39
Columns: ['destination', 'hotel_count', 'room_count', 'min_hotel_price', 'avg_hotel_price', 'max_hotel_price', 'avg_allotment']


,destination,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
0,Agra,4,11,5.33,35.862273,60.95,16.000000
1,Ahmedabad,7,35,26.53,107.852500,274.77,16.558824
2,Alappuzha,2,4,73.55,187.735556,335.48,12.222222
3,Amritsar,6,13,21.21,6278.796212,94287.82,5.303030
4,Bengaluru,18,68,15.89,106.213431,783.04,19.799020
5,Bhopal,2,9,27.32,39.246250,54.16,7.541667
6,Bhubaneswar,3,20,24.76,77.970000,248.48,10.470588
7,Chennai,9,39,24.24,65.692349,182.14,34.422819
8,Darjeeling,7,13,18.82,52.409785,112.87,5.688172
9,Delhi,40,82,14.74,159.123677,7642.72,12.740550


(39, 7)

In [18]:
# Merge accommodation features with the 50 destination records

integrated_df = destination_df.merge(
    accommodation_features,
    on="destination",
    how="left"
)

print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print("\nDestinations with accommodation:")
print(integrated_df["hotel_count"].notna().sum())

display(integrated_df.head())

Rows: 50
Columns: 18

Destinations with accommodation:
39


,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,sand_count,protected_area_count,hotel_count,room_count,min_hotel_price,avg_hotel_price,max_hotel_price,avg_allotment
0,Agra,13,60,98,42,32,3,1,0,0,0,0,4.0,11.0,5.33,35.862273,60.95,16.000000
1,Ahmedabad,12,52,40,14,2,0,2,0,0,0,0,7.0,35.0,26.53,107.852500,274.77,16.558824
2,Alappuzha,17,11,70,74,2,23,9,0,1,0,0,2.0,4.0,73.55,187.735556,335.48,12.222222
3,Amritsar,10,14,56,41,9,1,0,0,0,0,0,6.0,13.0,21.21,6278.796212,94287.82,5.303030
4,Andaman,0,0,0,0,2,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# Add weather features to the integrated dataset

integrated_df = integrated_df.merge(
    weather_df,
    on="destination",
    how="left"
)

print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print("\nDestinations with weather:")
print(integrated_df["temperature"].notna().sum())

Rows: 50
Columns: 32

Destinations with weather:
9


In [24]:
# Add destination-level flight features

integrated_df = integrated_df.merge(
    flight_features,
    on="destination",
    how="left"
)

print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print("\nDestinations with flight data:")
print(integrated_df["flight_count"].notna().sum())

Rows: 50
Columns: 39

Destinations with flight data:
8


In [26]:
integrated_df.head(50)

,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,...,visibility,rain_1h,timestamp,flight_count,min_flight_price,avg_flight_price,max_flight_price,avg_total_duration,avg_outbound_stops,avg_return_stops
0,Agra,13,60,98,42,32,3,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Ahmedabad,12,52,40,14,2,0,2,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Alappuzha,17,11,70,74,2,23,9,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amritsar,10,14,56,41,9,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman,0,0,0,0,2,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Bengaluru,97,39,99,62,38,0,16,0,0,...,NaN,NaN,NaN,13.0,8776.0,8776.0,8776.0,160.384615,0.0,0.000000
6,Bhopal,7,22,21,21,2,2,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Bhubaneswar,6,69,39,16,4,2,0,5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Chennai,62,43,96,74,8,9,9,0,9,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Coorg,0,4,0,0,2,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
# Check how much data is missing after integration

missing = integrated_df.isna().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print("Columns with missing values:")
print(missing)

Columns with missing values:
avg_total_duration     42
avg_return_stops       42
visibility             42
flight_count           42
min_flight_price       42
avg_flight_price       42
max_flight_price       42
avg_outbound_stops     42
longitude              41
temperature            41
country                41
rain_1h                41
weather_description    41
weather_condition      41
cloudiness             41
pressure               41
wind_speed             41
latitude               41
feels_like             41
humidity               41
timestamp              41
max_hotel_price        11
avg_hotel_price        11
room_count             11
hotel_count            11
avg_allotment          11
min_hotel_price        11
dtype: int64


In [28]:
# Create indicators showing whether each data source is available

integrated_df["weather_available"] = (
    integrated_df["temperature"].notna().astype(int)
)

integrated_df["accommodation_available"] = (
    integrated_df["hotel_count"].notna().astype(int)
)

integrated_df["flight_available"] = (
    integrated_df["flight_count"].notna().astype(int)
)

print("Weather available:",
      integrated_df["weather_available"].sum())

print("Accommodation available:",
      integrated_df["accommodation_available"].sum())

print("Flights available:",
      integrated_df["flight_available"].sum())

Weather available: 9
Accommodation available: 39
Flights available: 8


In [29]:
# Check the final integrated dataset

print("Rows:", len(integrated_df))
print("Columns:", len(integrated_df.columns))

print("\nDuplicate rows:", integrated_df.duplicated().sum())

print("\nAvailability:")
print(
    integrated_df[
        [
            "weather_available",
            "accommodation_available",
            "flight_available"
        ]
    ].sum()
)

display(integrated_df.head())

Rows: 50
Columns: 42

Duplicate rows: 0

Availability:
weather_available           9
accommodation_available    39
flight_available            8
dtype: int64


,destination,sight_count,park_count,restaurant_count,water_count,forest_count,wetland_count,river_count,mountain_count,coastal_count,...,flight_count,min_flight_price,avg_flight_price,max_flight_price,avg_total_duration,avg_outbound_stops,avg_return_stops,weather_available,accommodation_available,flight_available
0,Agra,13,60,98,42,32,3,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
1,Ahmedabad,12,52,40,14,2,0,2,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
2,Alappuzha,17,11,70,74,2,23,9,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
3,Amritsar,10,14,56,41,9,1,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0
4,Andaman,0,0,0,0,2,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [30]:
# Display all integrated dataset columns

for i, column in enumerate(integrated_df.columns, 1):
    print(i, column)

1 destination
2 sight_count
3 park_count
4 restaurant_count
5 water_count
6 forest_count
7 wetland_count
8 river_count
9 mountain_count
10 coastal_count
11 sand_count
12 protected_area_count
13 hotel_count
14 room_count
15 min_hotel_price
16 avg_hotel_price
17 max_hotel_price
18 avg_allotment
19 country
20 latitude
21 longitude
22 temperature
23 feels_like
24 humidity
25 pressure
26 wind_speed
27 cloudiness
28 weather_condition
29 weather_description
30 visibility
31 rain_1h
32 timestamp
33 flight_count
34 min_flight_price
35 avg_flight_price
36 max_flight_price
37 avg_total_duration
38 avg_outbound_stops
39 avg_return_stops
40 weather_available
41 accommodation_available
42 flight_available


In [31]:
# Check data types of all integrated features

print(integrated_df.dtypes)

destination                    str
sight_count                  int64
park_count                   int64
restaurant_count             int64
water_count                  int64
forest_count                 int64
wetland_count                int64
river_count                  int64
mountain_count               int64
coastal_count                int64
sand_count                   int64
protected_area_count         int64
hotel_count                float64
room_count                 float64
min_hotel_price            float64
avg_hotel_price            float64
max_hotel_price            float64
avg_allotment              float64
country                        str
latitude                   float64
longitude                  float64
temperature                float64
feels_like                 float64
humidity                   float64
pressure                   float64
wind_speed                 float64
cloudiness                 float64
weather_condition              str
weather_description 

In [32]:
# Convert timestamp from text to datetime

integrated_df["timestamp"] = pd.to_datetime(
    integrated_df["timestamp"],
    errors="coerce"
)

print(integrated_df["timestamp"].dtype)

datetime64[us, UTC]


In [33]:
# Inspect the timestamp values

print("Missing timestamps:", integrated_df["timestamp"].isna().sum())

display(
    integrated_df[
        ["destination", "timestamp"]
    ].head(10)
)

Missing timestamps: 45


,destination,timestamp
0,Agra,NaT
1,Ahmedabad,NaT
2,Alappuzha,NaT
3,Amritsar,NaT
4,Andaman,NaT
5,Bengaluru,NaT
6,Bhopal,NaT
7,Bhubaneswar,NaT
8,Chennai,NaT
9,Coorg,NaT


In [34]:
# Timestamp is API collection time, so it is not useful for destination modeling

integrated_df = integrated_df.drop(columns=["timestamp"])

print("Columns after removing timestamp:", len(integrated_df.columns))

Columns after removing timestamp: 41


In [35]:
# Check unique weather categories

print("Weather conditions:")
print(integrated_df["weather_condition"].value_counts(dropna=False))

print("\nWeather descriptions:")
print(integrated_df["weather_description"].value_counts(dropna=False))

Weather conditions:
weather_condition
NaN       41
Clouds     6
Rain       2
Clear      1
Name: count, dtype: int64

Weather descriptions:
weather_description
NaN                41
overcast clouds     5
light rain          2
few clouds          1
clear sky           1
Name: count, dtype: int64


In [36]:
# Keep the simpler weather category and remove the redundant description

integrated_df = integrated_df.drop(
    columns=["weather_description"]
)

print("Columns:", len(integrated_df.columns))

Columns: 40


In [37]:
# Check the categorical columns before encoding

print("Countries:")
print(integrated_df["country"].value_counts(dropna=False))

print("\nWeather conditions:")
print(integrated_df["weather_condition"].value_counts(dropna=False))

Countries:
country
NaN    41
IN      9
Name: count, dtype: int64

Weather conditions:
weather_condition
NaN       41
Clouds     6
Rain       2
Clear      1
Name: count, dtype: int64


In [38]:
# Country has the same value (India) for all available weather records,
# so it does not provide useful variation for the model.

integrated_df = integrated_df.drop(
    columns=["country"]
)

print("Columns:", len(integrated_df.columns))

Columns: 39


In [39]:
# Check which numerical columns still contain missing values

numeric_columns = integrated_df.select_dtypes(
    include=["int64", "float64"]
).columns

missing_numeric = integrated_df[numeric_columns].isna().sum()

print(
    missing_numeric[missing_numeric > 0]
    .sort_values(ascending=False)
)

min_flight_price      42
avg_flight_price      42
max_flight_price      42
avg_outbound_stops    42
avg_total_duration    42
avg_return_stops      42
visibility            42
flight_count          42
cloudiness            41
pressure              41
wind_speed            41
temperature           41
feels_like            41
latitude              41
humidity              41
longitude             41
rain_1h               41
avg_allotment         11
max_hotel_price       11
hotel_count           11
room_count            11
avg_hotel_price       11
min_hotel_price       11
dtype: int64


In [40]:
# Check whether our existing places data contains coordinates

places_df = pd.read_csv(
    "../data/cleaned/places_cleaned.csv"
)

print("Columns:")
print(places_df.columns.tolist())

Columns:
['query_category', 'search_radius_km', 'name', 'country', 'state', 'city', 'latitude', 'longitude', 'categories', 'formatted_address', 'place_id', 'limit_reached', 'destination_id', 'destination']


In [41]:
# Get one latitude/longitude pair for each destination

destination_coordinates = (
    places_df[
        ["destination", "latitude", "longitude"]
    ]
    .dropna()
    .drop_duplicates("destination")
)

print("Destinations with coordinates:",
      len(destination_coordinates))

display(destination_coordinates.head())

Destinations with coordinates: 50


,destination,latitude,longitude
0,Goa,15.181434,74.000957
113,Munnar,9.957520,77.052333
256,Manali,32.198870,77.202306
447,Jaipur,26.942913,75.824661
651,Udaipur,24.600200,73.680513


In [42]:
# Fill missing coordinates using our existing Places dataset

integrated_df = integrated_df.drop(
    columns=["latitude", "longitude"]
).merge(
    destination_coordinates,
    on="destination",
    how="left"
)

print("Missing latitude:", integrated_df["latitude"].isna().sum())
print("Missing longitude:", integrated_df["longitude"].isna().sum())

Missing latitude: 0
Missing longitude: 0


In [50]:
# Check remaining missing values after filling coordinates

missing = integrated_df.isna().sum()

print(
    missing[missing > 0]
    .sort_values(ascending=False)
)

avg_outbound_stops    42
max_flight_price      42
avg_flight_price      42
min_flight_price      42
flight_count          42
avg_return_stops      42
avg_total_duration    42
max_hotel_price       11
room_count            11
hotel_count           11
avg_hotel_price       11
min_hotel_price       11
avg_allotment         11
dtype: int64


In [44]:
# Inspect basic statistics of the numerical features

numeric_columns = integrated_df.select_dtypes(
    include=["int64", "float64"]
).columns

display(
    integrated_df[numeric_columns].describe().T
)

,count,mean,std,min,25%,50%,75%,max
sight_count,50.0,17.560000,27.069947,0.000000,3.000000,7.500000,12.750000,100.000000
park_count,50.0,26.680000,24.537718,0.000000,4.000000,19.500000,42.750000,78.000000
restaurant_count,50.0,55.520000,36.703022,0.000000,25.500000,56.000000,92.000000,100.000000
water_count,50.0,31.580000,29.512426,0.000000,9.000000,20.500000,45.000000,98.000000
forest_count,50.0,21.760000,25.083176,0.000000,3.250000,11.000000,31.750000,97.000000
wetland_count,50.0,4.860000,9.952971,0.000000,0.000000,1.000000,4.000000,41.000000
river_count,50.0,3.900000,4.726175,0.000000,0.000000,2.000000,6.750000,17.000000
mountain_count,50.0,2.180000,3.804884,0.000000,0.000000,1.000000,2.750000,21.000000
coastal_count,50.0,0.940000,2.721119,0.000000,0.000000,0.000000,0.000000,12.000000
sand_count,50.0,1.180000,5.516913,0.000000,0.000000,0.000000,0.000000,34.000000


In [45]:
# Reload the newly completed weather dataset.
# We keep weather_complete.csv untouched as the original 9-row file.

weather_df = pd.read_csv(
    "../data/cleaned/weather_data.csv"
)

print("Weather rows:", len(weather_df))
print(
    "Unique weather destinations:",
    weather_df["destination"].nunique()
)

Weather rows: 50
Unique weather destinations: 50


In [46]:
# Remove the old weather columns from integrated_df
# so we can merge the newly completed 50-destination weather data.

weather_columns = [
    "country",
    "latitude",
    "longitude",
    "temperature",
    "feels_like",
    "humidity",
    "pressure",
    "wind_speed",
    "cloudiness",
    "weather_condition",
    "weather_description",
    "visibility",
    "rain_1h",
    "timestamp"
]

existing_weather_columns = [
    col for col in weather_columns
    if col in integrated_df.columns
]

integrated_df = integrated_df.drop(
    columns=existing_weather_columns
)

print("Removed old weather columns:", len(existing_weather_columns))

Removed old weather columns: 11


In [47]:
# Merge the complete weather dataset using destination as the key.

integrated_df = integrated_df.merge(
    weather_df[["destination"] + weather_columns],
    on="destination",
    how="left"
)

print("Integrated shape:", integrated_df.shape)

Integrated shape: (50, 42)


In [48]:
# Check whether every destination now has weather data.

print(
    "Missing weather destinations:",
    integrated_df["temperature"].isna().sum()
)

print(
    "Weather destinations:",
    integrated_df["destination"]
    .loc[integrated_df["temperature"].notna()]
    .nunique()
)

Missing weather destinations: 0
Weather destinations: 50


In [49]:
# Check accommodation coverage after integrating the datasets

print(
    "Accommodation destinations:",
    integrated_df["accommodation_available"].sum()
)

print(
    "Missing accommodation destinations:",
    (
        integrated_df["accommodation_available"] == 0
    ).sum()
)

print("\nDestinations without accommodation:")
display(
    integrated_df.loc[
        integrated_df["accommodation_available"] == 0,
        ["destination"]
    ]
)

Accommodation destinations: 39
Missing accommodation destinations: 11

Destinations without accommodation:


,destination
4,Andaman
9,Coorg
21,Jim Corbett
23,Kaziranga
25,Kodaikanal
27,Ladakh
28,Mahabalipuram
36,Pahalgam
39,Ranchi
42,Shillong


In [51]:
# Check flight coverage in the integrated dataset

print(
    "Flight destinations:",
    integrated_df["flight_available"].sum()
)

print(
    "Missing flight destinations:",
    (
        integrated_df["flight_available"] == 0
    ).sum()
)

print("\nDestinations with flight data:")

display(
    integrated_df.loc[
        integrated_df["flight_available"] == 1,
        [
            "destination",
            "flight_count",
            "min_flight_price",
            "avg_flight_price",
            "max_flight_price"
        ]
    ]
)

Flight destinations: 8
Missing flight destinations: 42

Destinations with flight data:


,destination,flight_count,min_flight_price,avg_flight_price,max_flight_price
5,Bengaluru,13.0,8776.0,8776.0,8776.0
11,Delhi,29.0,14785.0,14785.0,14785.0
14,Goa,4.0,9538.0,9538.0,9538.0
19,Jaipur,3.0,15767.0,15767.0,15767.0
30,Mumbai,17.0,12057.0,12057.0,12057.0
31,Munnar,18.0,14132.0,14132.0,14132.0
44,Srinagar,3.0,18867.0,18867.0,18867.0
46,Udaipur,3.0,20498.0,20498.0,20498.0


In [52]:
# Create availability indicators for each external data source.
#
# 1 = data is available for this destination
# 0 = data was not collected / is unavailable
#
# These flags are important because missing API data is different
# from a real value of zero.

integrated_df["weather_available"] = (
    integrated_df["temperature"].notna().astype(int)
)

integrated_df["accommodation_available"] = (
    integrated_df["hotel_count"].notna().astype(int)
)

integrated_df["flight_available"] = (
    integrated_df["flight_count"].notna().astype(int)
)

print("Weather available:")
print(integrated_df["weather_available"].value_counts())

print("\nAccommodation available:")
print(integrated_df["accommodation_available"].value_counts())

print("\nFlights available:")
print(integrated_df["flight_available"].value_counts())

Weather available:
weather_available
1    50
Name: count, dtype: int64

Accommodation available:
accommodation_available
1    39
0    11
Name: count, dtype: int64

Flights available:
flight_available
0    42
1     8
Name: count, dtype: int64


In [53]:
# Final validation of the integrated dataset.
# We check structure and data availability before saving.

print("Rows:", integrated_df.shape[0])
print("Columns:", integrated_df.shape[1])

print(
    "\nDuplicate destinations:",
    integrated_df["destination"].duplicated().sum()
)

print("\nAvailability:")
print(
    integrated_df[
        [
            "weather_available",
            "accommodation_available",
            "flight_available"
        ]
    ].sum()
)

print("\nMissing values by column:")
missing_values = integrated_df.isna().sum()

display(
    missing_values[missing_values > 0]
)

Rows: 50
Columns: 42

Duplicate destinations: 0

Availability:
weather_available          50
accommodation_available    39
flight_available            8
dtype: int64

Missing values by column:


hotel_count           11
room_count            11
min_hotel_price       11
avg_hotel_price       11
max_hotel_price       11
avg_allotment         11
flight_count          42
min_flight_price      42
avg_flight_price      42
max_flight_price      42
avg_total_duration    42
avg_outbound_stops    42
avg_return_stops      42
dtype: int64

In [54]:
# Save the integrated dataset.
# Missing API values are intentionally preserved as NaN.
# They will be handled later during ML preprocessing.

integrated_output_path = "../data/cleaned/integrated_travel_dataset.csv"

integrated_df.to_csv(
    integrated_output_path,
    index=False
)

print(f"Integrated dataset saved to: {integrated_output_path}")
print("Rows:", integrated_df.shape[0])
print("Columns:", integrated_df.shape[1])

Integrated dataset saved to: ../data/cleaned/integrated_travel_dataset.csv
Rows: 50
Columns: 42


In [55]:
# Reload the saved dataset to make sure the file was written correctly.

integration_check = pd.read_csv(
    integrated_output_path
)

print("Rows:", integration_check.shape[0])
print("Columns:", integration_check.shape[1])

print(
    "Duplicate destinations:",
    integration_check["destination"].duplicated().sum()
)

print("\nAvailability:")
print(
    integration_check[
        [
            "weather_available",
            "accommodation_available",
            "flight_available"
        ]
    ].sum()
)

Rows: 50
Columns: 42
Duplicate destinations: 0

Availability:
weather_available          50
accommodation_available    39
flight_available            8
dtype: int64
